In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)

In [ ]:
from PIL import Image
from great_tables import GT, html, style, loc
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.constants import UNDEFINED, DM_ALONE, HTN_ALONE, HIV_ALONE, HTN_DM
from intecomm_analytics.utils import get_primary_cohorts_by_categorical_column, get_primary_cohorts_for_continuous_var, get_bp, get_glucose
from intecomm_rando.constants import COMMUNITY_ARM, FACILITY_ARM
from edc_pdutils.dataframes import get_subject_visit

In [ ]:
df_main = get_df_main_1858(None)

In [ ]:
df_visit = get_subject_visit("intecomm_subject.subjectvisit", normalize=True, localize=True)
df = df_visit[df_visit.visit_code_sequence==0].groupby(by=["subject_identifier"]).size().to_frame().reset_index()
df.columns = ["subject_identifier", "scheduled_visit_count"]
df_main = df_main.merge(df, on=["subject_identifier"], how="left")

df_visit = get_subject_visit("intecomm_subject.subjectvisit", normalize=True, localize=True)
df = df_visit[df_visit.visit_code_sequence>0].groupby(by=["subject_identifier"]).size().to_frame().reset_index()
df.columns = ["subject_identifier", "unscheduled_visit_count"]
df_main = df_main.merge(df, on=["subject_identifier"], how="left")
df_main = df_main.fillna({"unscheduled_visit_count": 0})

In [ ]:
# visit_count
tbl_dct = get_primary_cohorts_for_continuous_var(df_main, "scheduled_visit_count", ["median", "mean"])
dftbl = pd.DataFrame(tbl_dct)
dftbl["Statistics"] = dftbl["Statistics"].map({"count": "n", "mean": "Mean, SD", "median": "Median, range"})
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, range", "Mean, SD"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
df_scheduled_visit_count= dftbl.copy()

In [ ]:
# visit_count
tbl_dct = get_primary_cohorts_for_continuous_var(df_main, "unscheduled_visit_count", ["median", "mean"])
dftbl = pd.DataFrame(tbl_dct)
dftbl["Statistics"] = dftbl["Statistics"].map({"count": "n", "mean": "Mean, SD", "median": "Median, range"})
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, range", "Mean, SD"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
df_unscheduled_visit_count= dftbl.copy()

In [ ]:
df_scheduled_visit_count

In [ ]:
df_unscheduled_visit_count

In [ ]:
# primary_cohort
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "primary_cohort")
dftbl = pd.DataFrame(tbl_dct)
mapping = {DM_ALONE:"Diabetes alone", HTN_ALONE:"Hypertension alone", HTN_DM:"Diabetes and hypertension", HIV_ALONE:"HIV alone", UNDEFINED:"UNDEFINED", "n":"n"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl = dftbl[dftbl["Statistics"]!="UNDEFINED"]
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Diabetes alone", "Hypertension alone", "Diabetes and hypertension", "HIV alone"], ordered=True)
dftbl.sort_values(by=["Statistics"], ascending=True, inplace=True)
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dfnum = dftbl.iloc[0:1]
# dftbl["variable"] = "primary_cohort"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfcond = dftbl.copy()

In [ ]:
# country
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "country")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", "TZ": "Tanzania", "UG": "Uganda"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Tanzania", "Uganda"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "country"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfcountry = dftbl.copy()

In [ ]:
# gender
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "gender")
dftbl = pd.DataFrame(tbl_dct)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Female", "Male"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "gender"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfgender = dftbl.copy()

In [ ]:
# education
df1 = df_main.copy()
df1["education"] = df1["education"].apply(lambda x: "missing" if pd.isna(x) else x)
mapping = {
    "n": "n",
    "no_formal_education": "no_formal_education",
    "primary": "primary",
    "secondary": "secondary_or_tertiary",
    "post_secondary": "secondary_or_tertiary",
    "tertiary": "secondary_or_tertiary",
    "missing": "missing"}
df1["education"] = df1["education"].map(mapping)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "education")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "no_formal_education": "No formal education",
    "primary": "Primary",
    "secondary_or_tertiary": "Secondary or tertiary",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "No formal education", "Primary", "Secondary or tertiary", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "education"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfed = dftbl.copy()

In [ ]:
# age_in_years
df1 = df_main.copy()
bins = [0,34, 49, 110]
labels = ["<35", "35-49", ">=50"]
df1["age"] = pd.cut(df1["age_in_years"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "age")
dftbl = pd.DataFrame(tbl_dct)
data = ["Mean, SD"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        data.append(
            f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['age_in_years'].mean(),1)} "
            f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['age_in_years'].std(),1)})"
        )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Mean, SD", "<35", "35-49", ">=50"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "age_in_years"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfage= dftbl.copy()

In [ ]:
# smoker
df1 = df_main.copy()
df1["smoking_status"] = df1["smoking_status"].apply(lambda x: "missing" if pd.isna(x) else x)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "smoking_status")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "nonsmoker": "Non-smoker",
    "former_smoker": "Former smoker",
    "smoker": "Smoker",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Non-smoker", "Former smoker", "Smoker", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "smoker"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfsmoke = dftbl.copy()

In [ ]:
# marital status
df1 = df_main.copy()
df1["marital_status"] = df1["marital_status"].apply(lambda x: "missing" if pd.isna(x) else x)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "marital_status")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "married": "Married",
    "widowed": "Widowed",
    "divorced": "Divorced",
    "single":"Single",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Single", "Married", "Divorced", "Widowed", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "marital_status"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfmarital = dftbl.copy()

In [ ]:

# bp_controlled_endline
df1 = df_main.copy()
cond = ((df1.primary_cohort==HTN_ALONE) | (df1.primary_cohort==HTN_DM))
label = "<140/90 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension"
dftbl= get_bp(df1, "bp_controlled_endline", cond, label)
df_htn_bp = dftbl.copy()

In [ ]:
# bp_controlled_endline (alone)
df1 = df_main.copy()
col= "bp_controlled_endline"
cond = (df1.primary_cohort==HTN_ALONE)
label = "<140/90 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension alone"
dftbl= get_bp(df1, col, cond, label)
df_htn_alone_bp = dftbl.copy()

In [ ]:

# glucose_controlled_endline alone
df1 = df_main.copy()
col = "glucose_controlled_endline"
cond = (df1.primary_cohort==DM_ALONE)
label = "<7.0 mmol/L among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;diabetes alone"
dftbl= get_glucose(df1, col, cond, label)
df_dm_alone_glu = dftbl.copy()

In [ ]:
# glucose_controlled_endline
df1 = df_main.copy()
col = "glucose_controlled_endline"
cond = ((df1.primary_cohort==DM_ALONE) | (df1.primary_cohort==HTN_DM))
label = "<7.0 mmol/L among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;diabetes"
dftbl= get_glucose(df1, col, cond, label)
df_dm_glu = dftbl.copy()

In [ ]:
# bp_glucose_composite
df1 = df_main.copy()
cond = ((df1.primary_cohort==DM_ALONE) | (df1.primary_cohort==HTN_ALONE) | (df1.primary_cohort==HTN_DM))
label = "BP/Glucose controlled composite"
df1.loc[cond, "primary_controlled"] = df1.loc[cond, "primary_controlled"].fillna(-1)
tbl_dct = get_primary_cohorts_by_categorical_column(df1[cond], "primary_controlled")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Ncd", "Facility Ncd"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfcomposite = dftbl.copy()

In [ ]:
# vl_controlled_endline
df1 = df_main.copy()
col = "vl_controlled_endline"
cond = ((df1.hiv==1) & (df1.dm==0) & (df1.htn==0))
label = "<1000 copies per mL"
df1.loc[cond, col] = df1.loc[cond, col].fillna(-1)
tbl_dct = get_primary_cohorts_by_categorical_column(df1[cond], col)
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Hiv only", "Facility Hiv only"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfvl = dftbl.copy()

In [ ]:
# vl_controlled_endline_400
df1 = df_main.copy()
col = "vl_controlled_endline_400"
cond = ((df1.hiv==1) & (df1.dm==0) & (df1.htn==0))
label = "<400 copies per mL"
df1.loc[cond, col] = df1.loc[cond, col].fillna(-1)
tbl_dct = get_primary_cohorts_by_categorical_column(df1[cond],col)
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Hiv only", "Facility Hiv only"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfvl400 = dftbl.copy()

In [ ]:
# vl_controlled_endline_50
df1 = df_main.copy()
col = "vl_controlled_endline_50"
cond = ((df1.hiv==1) & (df1.dm==0) & (df1.htn==0))
label = "<50 copies per mL"
df1.loc[cond,col] = df1.loc[cond, col].fillna(-1)
tbl_dct = get_primary_cohorts_by_categorical_column(df1[cond], col)
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Hiv only", "Facility Hiv only"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfvl50 = dftbl.copy()

In [ ]:
dfvlall = pd.concat([dfvl, dfvl400, dfvl50])
dfvlall = dfvlall.reset_index(drop=True)

In [ ]:
# years since diagnosis
tbl_dct = get_primary_cohorts_for_continuous_var(df_main, "years_since_dx", ["median_iqr", "mean"])
dftbl = pd.DataFrame(tbl_dct)
dftbl["Statistics"] = dftbl["Statistics"].map({"count": "n", "mean": "Mean, SD", "median_iqr": "Median, IQR"})
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, IQR", "Mean, SD"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfyearsdx= dftbl.copy()

In [ ]:
groupings = [
    (dfnum, [""]),
    (dfcountry, ["Site"]),
    (dfcond, ["Condition"]),
    (dfyearsdx, ["Years since diagnosis"]),
    (dfgender, ["Sex"]),
    (dfage, ["Age"]),
    (dfmarital, ["Marital status"]),
    (dfed, ["Education"]),
    (dfsmoke, ["Smoking"]),
    (dfvlall, ["HIV viral load"]),
    (df_htn_alone_bp, ["Blood pressure"]),
    (df_htn_bp, ["Blood pressure"]),
    (df_dm_alone_glu, ["Fasting blood glucose"]),
    (df_dm_glu, ["Fasting blood glucose"]),
    (dfcomposite, ["BP/Glucose controlled composite"])
]

In [ ]:
group_row_headers = [(df, row_headers * (len(df))) for df, row_headers in groupings]
group_row_headers = [row_heading for _, row_headers in group_row_headers for row_heading in row_headers]

In [ ]:
# concat all
dftbl_final = pd.concat([df for df, _ in groupings])
dftbl_final = dftbl_final.reset_index(drop=True)

In [ ]:
# convert to GT
tbl_groups = dftbl_final.assign(group=group_row_headers)
tbl_groups = tbl_groups[tbl_groups['group'] != ""]
tbl_groups['Statistics'] = tbl_groups['Statistics'].apply(lambda x: f'&nbsp;&nbsp;&nbsp;{x}')
table = (GT(tbl_groups)
    .tab_header(title="Table 1: Baseline characteristics")
    .tab_spanner(label=html(f"Participants with diabetes,<BR>hypertension, or both<br> (n={dftbl_final.loc[0, ["Community Ncd", "Facility Ncd"]].sum()})"), columns=[1,2])
    .tab_spanner(label=html(f'Participants with<BR>HIV alone<BR>(n={dftbl_final.loc[0, ["Community Hiv only", "Facility Hiv only"]].sum()})'), columns=[3,4])
    .cols_label({
        "Community Ncd": html(f"Community<BR>(n={dftbl_final.loc[0, ["Community Ncd"]].sum()})"),
        "Facility Ncd": html(f"Facility<br>(n={dftbl_final.loc[0, ["Facility Ncd"]].sum()})"),
        "Community Hiv only": html(f"Community<br>(n={dftbl_final.loc[0, ["Community Hiv only"]].sum()})"),
        "Facility Hiv only": html(f"Facility<br>(n={dftbl_final.loc[0, ["Facility Hiv only"]].sum()})")})
    .cols_align(align='left', columns=[0])
    .cols_align(align='center', columns=[1,2,3,4])
    .tab_stub(rowname_col="Statistics", groupname_col="group")
    .opt_stylize(style=3)
    .opt_row_striping(row_striping=False)
    .opt_vertical_padding(scale=1.2)
    .opt_horizontal_padding(scale=1.0)
    .tab_options(
        stub_background_color='white',
        row_group_border_bottom_style='hidden',
        row_group_padding=0.5,
        row_group_background_color="white",
        table_background_color="white",
        table_font_size=12)
    .tab_style(
        style=[style.fill(color="white"), style.text(color="black")],
        locations=loc.body(columns=[1,2, 3, 4], rows=list(range(0, len(dftbl_final)))))
    )
table.show()

In [ ]:
# save as png
table.save(analysis_folder / "baseline_characteristics.png")

In [ ]:
# export to PDF
image = Image.open(analysis_folder / "baseline_characteristics.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / "baseline_characteristics.pdf", "PDF", resolution=800, optimize=True, quality=95)